In [17]:
n_mol = 3
mol_type = "6r7m"
in_out_folder = "../Simulations/continuous_simulations/$(n_mol)_$(mol_type)/"
simulation_time_minutes = 14 * 60.0

840.0

In [18]:
files = filter(x -> occursin(".jld2", x), readdir("../../$(in_out_folder)"))

12-element Vector{String}:
 "1.jld2"
 "10.jld2"
 "11.jld2"
 "12.jld2"
 "2.jld2"
 "3.jld2"
 "4.jld2"
 "5.jld2"
 "6.jld2"
 "7.jld2"
 "8.jld2"
 "9.jld2"

In [19]:
input_specifier = "continuous_$(n_mol)_$(mol_type)"
open("../configs/$(input_specifier)_config.txt", "w") do io
    i = 1
    println(io,"ArrayTaskID input_string")
    for file in files
        input_string = escape_string("file=\"$file\";simulation_time_minutes=$simulation_time_minutes;in_out_folder=\"$in_out_folder\"")
        println(io, "$i $input_string")
        i += 1
    end
end

In [20]:
total_simulations = length(readlines("../configs/$(input_specifier)_config.txt")) - 1

12

In [21]:
hours = Int(round(simulation_time_minutes / 60.0))
days = hours ÷ 24
remaining_hours = hours % 24
remaining_hours_string = remaining_hours < 10 ? "0$(remaining_hours)" : string(remaining_hours)
buffer_time_string = simulation_time_minutes < 5 ? "0$(Int(simulation_time_minutes)+2)" : "30"

open("../$(input_specifier)_script.job", "w") do io
    println(io, "#!/bin/bash")
    println(io, "#SBATCH --job-name=SolvationSimulations")
    println(io, "#SBATCH --time=0$(days)-$(remaining_hours_string):$(buffer_time_string)")
    println(io, "#SBATCH --ntasks=1")
    println(io, "#SBATCH --cpus-per-task=1")
    println(io, "#SBATCH --array=1-$(total_simulations)")
    println(io, "#SBATCH --chdir=/work/spirandelli/MorphoMolHPC/")
    println(io, "#SBATCH -o ./job_log/$(input_specifier)/%a.out # STDOUT")
    println(io, "")
    println(io, "export http_proxy=http://proxy2.uni-potsdam.de:3128 #Setting proxy, due to lack of Internet on compute nodes.")
    println(io, "export https_proxy=http://proxy2.uni-potsdam.de:3128")
    println(io, "")
    println(io, "module purge")
    println(io, "module load lang/Julia/1.7.3-linux-x86_64")
    println(io, "")
    println(io, "# Specify the path to the config file")
    println(io, "config=hpc_scripts/configs/$(input_specifier)_config.txt")
    println(io, "")
    println(io, "# Extract the variables from config file")
    println(io, "config_string=\$(awk -v ArrayTaskID=\$SLURM_ARRAY_TASK_ID '\$1==ArrayTaskID {print \$2}' \$config)")
    println(io, "")
    println(io, "julia -e \"$(escape_string("include(\"julia_scripts/continuous_cc_rwm_ma_call.jl\"); continuous_cc_rwm_ma_call(\"\$config_string\")"))\"")
end